In [46]:
import pandas as pd
import numpy as np

# Cargar precios ajustados
adj_close = pd.read_csv(
    "../Datos_csv/adj_close.csv",
    index_col=0,
    parse_dates=True
)

# Cargar volumen
volume = pd.read_csv(
    "../Datos_csv/volume.csv",
    index_col=0,
    parse_dates=True
)


In [47]:
to_drop = ['BIL', 'EWY', 'GLD', 'IAU', 'SHY']

adj_close = adj_close.drop(columns=to_drop)
volume    = volume.drop(columns=to_drop)


In [48]:
etf_class_map = {
    # 3.1.1 Mercado estadounidense (Core US)
    "SPY": "Core_US", "IVV": "Core_US", "VOO": "Core_US", "QQQ": "Core_US",
    "VTI": "Core_US", "ITOT": "Core_US", "DIA": "Core_US", "IWM": "Core_US",
    "IJR": "Core_US", "MDY": "Core_US",

    # 3.1.2 Factores y estilos
    "VUG": "Factor_Style", "IWF": "Factor_Style", "VTV": "Factor_Style",
    "IWD": "Factor_Style", "SCHD": "Factor_Style", "VIG": "Factor_Style",
    "DVY": "Factor_Style", "MTUM": "Factor_Style", "QUAL": "Factor_Style",
    "USMV": "Factor_Style", "VLUE": "Factor_Style",

    # 3.1.3 Sectores económicos
    "XLK": "Sector_economico", "XLF": "Sector_economico", "XLV": "Sector_economico", "XLY": "Sector_economico",
    "XLP": "Sector_economico", "XLE": "Sector_economico", "XLI": "Sector_economico", "XLU": "Sector_economico",
    "XLB": "Sector_economico", "XLRE": "Sector_economico",

    # 3.1.4 Mercados internacionales sin EEUU
    "VEA": "International", "IEFA": "International", "VWO": "International",
    "IEMG": "International", "EEM": "International", "EFA": "International",
    "EWJ": "International", "EWG": "International", "EWU": "International",
    "EWQ": "International", "INDA": "International", "EWZ": "International",
    "FXI": "International", "MCHI": "International", "EWT": "International",

    # 3.1.5 Renta fija (Bonos)
    "BND": "Bonds", "AGG": "Bonds", "IEF": "Bonds", "TLT": "Bonds",
    "LQD": "Bonds", "HYG": "Bonds", "JNK": "Bonds", "TIP": "Bonds",

    # 3.1.6 Materias primas y activos reales
    "SLV": "Real_Assets", "USO": "Real_Assets", "DBC": "Real_Assets",
    "VNQ": "Real_Assets"
}

In [49]:
adj_close_long = (
    adj_close
    .reset_index()
    .rename(columns={"index": "Date"})
    .melt(id_vars="Date", var_name="ETF", value_name="adj_close")
)

volume_long = (
    volume
    .reset_index()
    .rename(columns={"index": "Date"})
    .melt(id_vars="Date", var_name="ETF", value_name="volume")
)

# Unimos precios y volumen
df_semanal = adj_close_long.merge(volume_long, on=["Date", "ETF"], how="inner")


In [41]:
adj_close= adj_close["SPY"]

In [ ]:
# Check for NaN values in adj_close and volume before filtering
print("NaN in adj_close:", df_semanal["adj_close"].isna().sum())
print("NaN in volume:", df_semanal["volume"].isna().sum())
print("Volume <= 0:", (df_semanal["volume"] <= 0).sum())

# Apply filters to remove NaN and invalid volume
df_semanal = df_semanal[df_semanal["adj_close"].notna()].copy()
df_semanal = df_semanal[df_semanal["volume"].notna()].copy()
df_semanal = df_semanal[df_semanal["volume"] > 0].copy()

In [42]:
ret_log_diff = np.log(adj_close).diff()

In [43]:
# Eliminar la primera fila NaN generada por diff()
ret_log_diff = ret_log_diff.dropna()

In [45]:
ret_log_diff.index = pd.to_datetime(ret_log_diff.index)
iso_cal = ret_log_diff.index.isocalendar()

ret_log_diff["weekday"] = ret_log_diff.index.weekday
ret_log_diff["year"] = iso_cal.year.astype(int)
ret_log_diff["iso_week"] = iso_cal.week.astype(int)

ret_log_diff["week_key"] = (
    ret_log_diff["year"].astype(str)
    + "-W"
    + ret_log_diff["iso_week"].astype(str).str.zfill(2)
)


DateParseError: Unknown datetime string format, unable to parse: weekday

In [37]:
ret_log_diff.head()

Date
2021-01-05 00:00:00    0.006863
2021-01-06 00:00:00    0.005961
2021-01-07 00:00:00    0.014748
2021-01-08 00:00:00    0.005681
2021-01-11 00:00:00   -0.006763
Name: SPY, dtype: object

In [ ]:

# 4. Filtrar semanas con al menos 4 días de datos
week_counts = ret_log_diff_recortado.groupby("week_key").size()
valid_weeks = week_counts[week_counts >= 4].index

ret_log_diff_recortado_completo = ret_log_diff_recortado[ret_log_diff_recortado["week_key"].isin(valid_weeks)].copy()

In [6]:
week_counts

week_key
2021-W01    5
2021-W02    5
2021-W03    4
2021-W04    5
2021-W05    5
           ..
2026-W01    4
2026-W02    5
2026-W03    5
2026-W04    4
2026-W05    5
Length: 265, dtype: int64

In [ ]:

# Dataset semanal con MEDIA de precios
price_cols = adj_close.columns.tolist()

# fecha mínima de cada semana
weekly_dates = ret_log_diff_recortado_completo.groupby("week_key").apply(
    lambda df: pd.Series({
        "date_min": df.index.min(),
        "date_max": df.index.max(),
        "year": df["year"].iloc[0]
    })
)
# media semanal de cada activo
weekly_prices = ret_log_diff_recortado_completo.groupby("week_key")[price_cols].mean()

# unir fechas y precios
weekly = pd.concat([weekly_dates, weekly_prices], axis=1).reset_index(drop=True)

# crear identificador week1, week2, ...
weekly.insert(0, "week", [f"week{i}" for i in range(1, len(weekly) + 1)])

# resumen
print("Fecha mínima original:", ret_log_diff.index.min())
print("Fecha máxima original:", ret_log_diff.index.max())
print("Primer lunes usado:", first_monday)
print("Último viernes usado:", last_friday)
print("Número de semanas válidas:", len(weekly))

print("\nEjemplo dataset semanal:")
print(weekly.head())

Fecha mínima original: 2021-01-04 00:00:00
Fecha máxima original: 2026-02-02 00:00:00
Primer lunes usado: 2021-01-04 00:00:00
Último viernes usado: 2026-01-30 00:00:00
Número de semanas válidas: 265

Ejemplo dataset semanal:
    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...        XLB        XLE        XLF  \
0  281.092810  81.717987  47.349725  ...  33.924350  16.670438  27.635263   
1  283.641693  84.151309  48.508289  ...  34.625089  17.884007  28.631347   
2  284.459206  83.995672  49.746047  ...  34.015105  17.771009  28.160363   
3  279.793677  82.5207

In [8]:
weekly.columns

Index(['week', 'date_min', 'date_max', 'year', 'AGG', 'BND', 'DBC', 'DIA',
       'DVY', 'EEM', 'EFA', 'EWG', 'EWJ', 'EWQ', 'EWT', 'EWU', 'EWZ', 'FXI',
       'HYG', 'IEF', 'IEFA', 'IEMG', 'IJR', 'INDA', 'ITOT', 'IVV', 'IWD',
       'IWF', 'IWM', 'JNK', 'LQD', 'MCHI', 'MDY', 'MTUM', 'QQQ', 'QUAL',
       'SCHD', 'SLV', 'SPY', 'TIP', 'TLT', 'USMV', 'USO', 'VEA', 'VIG', 'VLUE',
       'VNQ', 'VOO', 'VTI', 'VTV', 'VUG', 'VWO', 'XLB', 'XLE', 'XLF', 'XLI',
       'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY'],
      dtype='str')

In [9]:
def crear_semanas_anteriores(df, lags=1):

    cols_no_etf = ["week", "date_min", "date_max","year"]
    etf_cols = [col for col in df.columns if col not in cols_no_etf]

    lag_dfs = []

    for lag in range(1, lags + 1):

        lag_df = df[etf_cols].shift(lag)
        lag_df = lag_df.add_suffix(f"_semana_{lag}")

        lag_dfs.append(lag_df)

    df_lag = pd.concat([df] + lag_dfs, axis=1)

    return df_lag

crear semana -1

In [10]:
weekly_semanal_1 = crear_semanas_anteriores(weekly, lags=1)

print(weekly_semanal_1.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_1  XLE_semana_1  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...     33.924350     16.670438   
2  284.459206  83.995672  49.746047  ...     34.625089     17.884007   
3  279.793677  82.520750  48.826846  ...     34.015105     17.771009   
4  281.648346  83.240074  49.465749  ...     32.634224     16.769775   

   XLF_semana_1  XLI_semana_1  XLK_semana_1  XLP_semana_1  XLRE_semana_1  \
0           NaN           NaN           NaN   

crear semana -2

In [11]:
weekly_semanal_2 = crear_semanas_anteriores(weekly, lags=2)

print(weekly_semanal_2.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_2  XLE_semana_2  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...     33.924350     16.670438   
3  279.793677  82.520750  48.826846  ...     34.625089     17.884007   
4  281.648346  83.240074  49.465749  ...     34.015105     17.771009   

   XLF_semana_2  XLI_semana_2  XLK_semana_2  XLP_semana_2  XLRE_semana_2  \
0           NaN           NaN           NaN   

In [12]:
weekly_semanal_3 = crear_semanas_anteriores(weekly, lags=3)

print(weekly_semanal_3.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_3  XLE_semana_3  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...     33.924350     16.670438   
4  281.648346  83.240074  49.465749  ...     34.625089     17.884007   

   XLF_semana_3  XLI_semana_3  XLK_semana_3  XLP_semana_3  XLRE_semana_3  \
0           NaN           NaN           NaN   

In [13]:
weekly_semanal_4 = crear_semanas_anteriores(weekly, lags=4)

print(weekly_semanal_4.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_4  XLE_semana_4  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...           NaN           NaN   
4  281.648346  83.240074  49.465749  ...      33.92435     16.670438   

   XLF_semana_4  XLI_semana_4  XLK_semana_4  XLP_semana_4  XLRE_semana_4  \
0           NaN           NaN           NaN   

In [14]:
weekly_semanal_5 = crear_semanas_anteriores(weekly, lags=5)

print(weekly_semanal_5.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_5  XLE_semana_5  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...           NaN           NaN   
4  281.648346  83.240074  49.465749  ...           NaN           NaN   

   XLF_semana_5  XLI_semana_5  XLK_semana_5  XLP_semana_5  XLRE_semana_5  \
0           NaN           NaN           NaN   

In [15]:
weekly_semanal_6 = crear_semanas_anteriores(weekly, lags=6)

print(weekly_semanal_6.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_6  XLE_semana_6  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...           NaN           NaN   
4  281.648346  83.240074  49.465749  ...           NaN           NaN   

   XLF_semana_6  XLI_semana_6  XLK_semana_6  XLP_semana_6  XLRE_semana_6  \
0           NaN           NaN           NaN   

In [16]:
weekly_semanal_7 = crear_semanas_anteriores(weekly, lags=7)

print(weekly_semanal_7.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_7  XLE_semana_7  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...           NaN           NaN   
4  281.648346  83.240074  49.465749  ...           NaN           NaN   

   XLF_semana_7  XLI_semana_7  XLK_semana_7  XLP_semana_7  XLRE_semana_7  \
0           NaN           NaN           NaN   

In [17]:
weekly_semanal_8 = crear_semanas_anteriores(weekly, lags=8)

print(weekly_semanal_8.head())

    week   date_min   date_max  year         AGG        BND        DBC  \
0  week1 2021-01-04 2021-01-08  2021  101.305956  75.248289  13.062511   
1  week2 2021-01-11 2021-01-15  2021  100.964594  74.942596  13.356618   
2  week3 2021-01-19 2021-01-22  2021  101.144323  75.063240  13.302234   
3  week4 2021-01-25 2021-01-29  2021  101.259404  75.164143  13.248720   
4  week5 2021-02-01 2021-02-05  2021  100.981580  74.936356  13.652465   

          DIA        DVY        EEM  ...  XLB_semana_8  XLE_semana_8  \
0  281.092810  81.717987  47.349725  ...           NaN           NaN   
1  283.641693  84.151309  48.508289  ...           NaN           NaN   
2  284.459206  83.995672  49.746047  ...           NaN           NaN   
3  279.793677  82.520750  48.826846  ...           NaN           NaN   
4  281.648346  83.240074  49.465749  ...           NaN           NaN   

   XLF_semana_8  XLI_semana_8  XLK_semana_8  XLP_semana_8  XLRE_semana_8  \
0           NaN           NaN           NaN   

In [18]:
import os

def guardar_dataset(df, nombre_archivo):
    
    out_dir = "../Datos_csv"
    os.makedirs(out_dir, exist_ok=True)

    path = os.path.join(out_dir, nombre_archivo)

    df.to_csv(path, index=False)

    print("Guardado:")
    print(" -", path, df.shape)

In [19]:
guardar_dataset(weekly, "df_semanal.csv")
guardar_dataset(weekly_semanal_1, "df_semanal_1.csv")
guardar_dataset(weekly_semanal_2, "df_semanal_2.csv")
guardar_dataset(weekly_semanal_3, "df_semanal_3.csv")
guardar_dataset(weekly_semanal_4, "df_semanal_4.csv")
guardar_dataset(weekly_semanal_5, "df_semanal_5.csv")
guardar_dataset(weekly_semanal_6, "df_semanal_6.csv")
guardar_dataset(weekly_semanal_7, "df_semanal_7.csv")
guardar_dataset(weekly_semanal_8, "df_semanal_8.csv")

Guardado:
 - ../Datos_csv\df_semanal.csv (265, 62)
Guardado:
 - ../Datos_csv\df_semanal_1.csv (265, 120)
Guardado:
 - ../Datos_csv\df_semanal_2.csv (265, 178)
Guardado:
 - ../Datos_csv\df_semanal_3.csv (265, 236)
Guardado:
 - ../Datos_csv\df_semanal_4.csv (265, 294)
Guardado:
 - ../Datos_csv\df_semanal_5.csv (265, 352)
Guardado:
 - ../Datos_csv\df_semanal_6.csv (265, 410)
Guardado:
 - ../Datos_csv\df_semanal_7.csv (265, 468)
Guardado:
 - ../Datos_csv\df_semanal_8.csv (265, 526)
